In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
import segmentation_models_pytorch as smp
import numpy as np
from tqdm import tqdm
import random

from models import CNNUnary

import data
import utils
import visu

In [ ]:
category = 15

train_dataset, test_dataset = data.single_cat_OxfordIIITPet(category)

In [ ]:
i = 0

image, (mask, cat) = train_dataset[i]
mask = utils.preprocess(mask)

visu.plot_mask(image, mask)

In [ ]:


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CATEGORY    = 15
NUM_CLASSES = 2
BATCH_SIZE  = 8
NUM_EPOCHS  = 5
LR          = 1e-4
VAL_SPLIT   = 0.2
IMG_SIZE    = 256
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKER = 0

image_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

mask_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE),
                      interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),          # keeps integer values
])

class PetForegroundDataset(Dataset):
    """
    Wraps OxfordIIITPet and converts the trimap mask to binary:
        1 (foreground) → 1
        2 (background) → 0
        3 (border)     → 255  (ignored in loss)
    """
    def __init__(self, dataset):
        self.base = dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        image, (mask, _) = self.base[i]

        image = image_tf(image)                  # [3, H, W]
        mask  = mask_tf(mask).squeeze(0).long()  # [H, W]  values: 1,2,3

        # trimap → binary  (border pixels → 255 = ignore)
        binary = torch.full_like(mask, 255)
        binary[mask == 1] = 1   # foreground
        binary[mask == 2] = 0   # background

        return image, binary


train_ds = PetForegroundDataset(train_dataset)
val_ds   = PetForegroundDataset(test_dataset)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKER)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKER)

model = CNNUnary().to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


def iou_score(preds, targets, num_classes=2, ignore=255):
    """Mean IoU over valid classes."""
    ious = []
    preds   = preds.view(-1)
    targets = targets.view(-1)
    valid   = targets != ignore
    preds, targets = preds[valid], targets[valid]
    for c in range(num_classes):
        inter = ((preds == c) & (targets == c)).sum().float()
        union = ((preds == c) | (targets == c)).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return np.mean(ious) if ious else 0.0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_iou, n = 0.0, 0.0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)

            logits = model(images)                       # [B, C, H, W]
            loss   = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)                 # [B, H, W]
            total_loss += loss.item()
            total_iou  += iou_score(preds.cpu(), masks.cpu())
            n += 1

    return total_loss / n, total_iou / n

best_val_iou  = 0.0
best_ckpt     = "best_unet.pth"

for epoch in tqdm(range(1, NUM_EPOCHS + 1)):
    train_loss, train_iou = run_epoch(train_loader, train=True)
    val_loss,   val_iou   = run_epoch(val_loader,   train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss:.4f}  mIoU: {train_iou:.4f} | "
          f"Val   loss: {val_loss:.4f}  mIoU: {val_iou:.4f}")

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), best_ckpt)
        print(f"Saved best model (val mIoU = {best_val_iou:.4f})")

print(f"\nTraining done. Best val mIoU: {best_val_iou:.4f}")

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
pass

In [ ]:
image, (mask, _) = test_dataset[7]
pred = model.predict(image)

visu.plot_mask(image, pred)